# Day 10 — CNN from scratch + transfer learning on CIFAR-10


In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = 'cuda' if torch.cuda.is_available() else 'cpu'
MEAN = (0.4914, 0.4822, 0.4465); STD = (0.2470, 0.2435, 0.2616)
train_tfm = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
test_tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
train_ds = datasets.CIFAR10('./data', train=True,  download=True, transform=train_tfm)
test_ds  = datasets.CIFAR10('./data', train=False, download=True, transform=test_tfm)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)
test_dl  = DataLoader(test_ds, batch_size=256, num_workers=2)


## Small CNN from scratch

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(64*8*8, 256), nn.ReLU(),
            nn.Linear(256, num_classes),
        )
    def forward(self, x): return self.head(self.features(x))

def train(model, epochs=5, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    for ep in range(epochs):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
        model.eval(); c = t = 0
        with torch.no_grad():
            for xb, yb in test_dl:
                xb, yb = xb.to(device), yb.to(device)
                c += (model(xb).argmax(1) == yb).sum().item(); t += len(yb)
        print(f'ep {ep}: test acc = {c/t:.4f}')

train(SmallCNN(), epochs=5)


## Transfer learning with ResNet18

In [ ]:
rn = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
rn.fc = nn.Linear(rn.fc.in_features, 10)
train(rn, epochs=5, lr=1e-3)
